In [39]:
import keras.backend as K
import os
import numpy as np
import pylab as plt
from keras.utils import to_categorical
from keras.models import Model
from keras.layers import Input
from keras.layers import LSTM
from keras.layers import Dense
from keras.layers.convolutional import Conv3D
from keras.layers.convolutional_recurrent import ConvLSTM2D
from keras.layers.normalization import BatchNormalization
from keras.models import load_model
from keras.callbacks import EarlyStopping
from keras.callbacks import ModelCheckpoint
import matplotlib.pyplot as plt
from keras.utils import to_categorical
from keras.regularizers import l2
from keras.backend import clip 
import math

In [40]:
"""
2-layer 
"""    
def define_models_2_precipitation(n_filter, filter_size):
    # define training encoder
    encoder_inputs = Input(shape=(None, 128, 110, 1))
    encoder_1 = ConvLSTM2D(filters = n_filter, kernel_size=filter_size, activation='relu', padding='same', return_sequences=True, return_state=True,
                           kernel_regularizer=l2(0.0005), recurrent_regularizer=l2(0.0005), bias_regularizer=l2(0.0005))
    encoder_2 = ConvLSTM2D(filters = n_filter, kernel_size=filter_size, activation='relu', padding='same', return_sequences=True, return_state=True,
                           kernel_regularizer=l2(0.0005), recurrent_regularizer=l2(0.0005), bias_regularizer=l2(0.0005))
    encoder_outputs_1, encoder_state_h_1, encoder_state_c_1 = encoder_1(encoder_inputs)
    encoder_outputs_2, encoder_state_h_2, encoder_state_c_2 = encoder_2(encoder_outputs_1)
    # define training decoder
    decoder_inputs = Input(shape=(None, 128, 110, 1))
    decoder_1 = ConvLSTM2D(filters=n_filter, kernel_size=filter_size, activation='relu', padding='same', return_sequences=True, return_state=True,
                           kernel_regularizer=l2(0.0005), recurrent_regularizer=l2(0.0005), bias_regularizer=l2(0.0005))
    decoder_2 = ConvLSTM2D(filters=n_filter, kernel_size=filter_size, activation='relu', padding='same', return_sequences=True, return_state=True,
                           kernel_regularizer=l2(0.0005), recurrent_regularizer=l2(0.0005), bias_regularizer=l2(0.0005))
    decoder_outputs_1, _, _ = decoder_1([decoder_inputs, encoder_state_h_1, encoder_state_c_1])
    decoder_outputs_2, _, _ = decoder_2([decoder_outputs_1, encoder_state_h_2, encoder_state_c_2])
    decoder_conv3d = Conv3D(filters=1, kernel_size=(1,1,64), activation='relu', padding='same', data_format='channels_last',
                            kernel_regularizer=l2(0.0005), bias_regularizer=l2(0.0005))
    decoder_outputs = decoder_conv3d(decoder_outputs_2)
    #clip(dec oder_outputs, 0, 255)
    
#    denselayer = Dense(1, activation='softmax')
#    decoder_outputs = denselayer(decoder_outputs)
    
    
    model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    #print(model.summary(line_length=250))
    
    # define inference encoder
    encoder_model = Model(encoder_inputs, [encoder_state_h_1, encoder_state_c_1, encoder_state_h_2, encoder_state_c_2])
    
    # define inference decoder
    decoder_state_input_h_1 = Input(shape=(128,110,n_filter))
    decoder_state_input_c_1 = Input(shape=(128,110,n_filter))
    decoder_state_input_h_2 = Input(shape=(128,110,n_filter))
    decoder_state_input_c_2 = Input(shape=(128,110,n_filter))
    decoder_output_1, decoder_state_h_1_new, decoder_state_c_1_new = decoder_1([decoder_inputs, decoder_state_input_h_1, decoder_state_input_c_1])
    decoder_output_2, decoder_state_h_2_new, decoder_state_c_2_new = decoder_2([decoder_output_1, decoder_state_input_h_2, decoder_state_input_c_2])
    decoder_output = decoder_conv3d(decoder_output_2)
    #clip(decoder_output, 0, 255)
    
#    decoder_output = denselayer(decoder_output)
    
    decoder_model = Model([decoder_inputs , decoder_state_input_h_1 , decoder_state_input_c_1, decoder_state_input_h_2 , decoder_state_input_c_2],
                          [decoder_output, decoder_state_h_1_new, decoder_state_c_1_new, decoder_state_h_2_new, decoder_state_c_2_new])
    
    return model, encoder_model, decoder_model


In [41]:
train_2_precipitation, infenc_2_precipitation, infdec_2_precipitation = define_models_2_precipitation(n_filter=64, filter_size=3)
train_2_precipitation.compile(loss='mse', optimizer='adam', metrics=['mse'])#, metrics=['mse'])
train_2_precipitation.load_weights('dump.h5')


In [42]:
def predict_sequence_2(infenc, infdec, source, n_steps):
    # encode
    state_h_1, state_c_1, state_h_2, state_c_2 = infenc.predict(source)  # source_dim = ()
    #decoder_input = source[:,-1,:,:,:].reshape((1,1,64,64,1))
    decoder_input = np.repeat(0,128*110).reshape((1,1,128,110,1))
    #decoder_input = decoder_input.astype('float')
    # 123
    output = list()
    for t in range(n_steps):
        # predict next char
        yhat, h_1, c_1, h_2, c_2 = infdec.predict([decoder_input, state_h_1, state_c_1, state_h_2, state_c_2])
        # store prediction
        output.append(yhat[0,0,:])
        # update state
        state_h_1, state_c_1, state_h_2, state_c_2 = h_1, c_1, h_2, c_2
        # update target sequence
        decoder_input = yhat
    return np.array(output)


In [43]:
data = np.load('data_with_frame_splits.npy')
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID" 
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

In [44]:
which = 599

#which =  np.random.randint(5485)
track = data[which,:16,:,:,:]
track.shape


(16, 128, 110, 1)

In [45]:
history = track[np.newaxis, ::, ::, ::, ::]
history.shape


(1, 16, 128, 110, 1)

In [46]:
import tensorflow as tf
tf.config.experimental.set_memory_growth = True

In [47]:

prediction_1 = predict_sequence_2(infenc_2_precipitation, infdec_2_precipitation, history, 7)
print(prediction_1.shape)

UnknownError:  Failed to get convolution algorithm. This is probably because cuDNN failed to initialize, so try looking to see if a warning log message was printed above.
	 [[node model_13/conv_lst_m2d_16/convolution (defined at <ipython-input-42-dc016310ee30>:3) ]] [Op:__inference_predict_function_10146]

Function call stack:
predict_function
